<a href="https://colab.research.google.com/github/T-Syam-Kumar/Hand_written-Digit-Recogniser/blob/main/HF_HWDC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ================================
# Handwritten Digit Classifier
# Draw with Mouse | Run in Colab
# ================================

# Install dependencies
!pip install gradio torch torchvision pillow numpy -q

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image, ImageOps
import gradio as gr
import numpy # Added numpy import

# ----------------
# CNN Model (MNIST)
# ----------------
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ----------------
# Load Pretrained Weights
# ----------------
# The mnist_cnn.pth file is now generated by a separate training cell.
# No need to download it here.

model = DigitClassifier()
model.load_state_dict(torch.load("mnist_cnn.pth", map_location="cpu"))
model.eval()

# ----------------
# Preprocessing
# ----------------
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

def preprocess(image):
    print(f"[DEBUG] Original image type: {type(image)}")
    if image is None:
        raise ValueError("Input image is None")
    # Convert numpy array to PIL Image if it's a numpy array
    if isinstance(image, numpy.ndarray):
        image = Image.fromarray(image)
        print(f"[DEBUG] Converted from numpy.ndarray to PIL Image: {type(image)}")
    image = image.convert("L")      # grayscale
    print(f"[DEBUG] After grayscale: {image.mode}, {image.size}")
    image = ImageOps.invert(image)  # invert (MNIST style)
    print(f"[DEBUG] After invert: {image.mode}, {image.size}")
    image = transform(image)
    print(f"[DEBUG] After transform: {image.shape}")
    image = image.unsqueeze(0)
    print(f"[DEBUG] After unsqueeze: {image.shape}")
    return image

# ----------------
# Prediction
# ----------------
def predict(image):
    print(f"[DEBUG] Inside predict function, received image: {type(image)}")
    try:
        image = preprocess(image)
        with torch.no_grad():
            output = model(image)
            pred = torch.argmax(output, dim=1).item()
        print(f"[DEBUG] Prediction successful: {pred}")
        return f"Predicted Digit: {pred}"
    except Exception as e:
        print(f"[ERROR] Prediction failed: {e}")
        return f"Error during prediction: {e}"

# ----------------
# Gradio UI
# ----------------
gr.Interface(
    fn=predict,
    inputs=gr.Image(),
    outputs="text",
    title="🖱️ Handwritten Digit Classifier",
    description="Draw a digit (0–9) using your mouse"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://122d34fd7a9095af22.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define the same DigitClassifier class to ensure compatibility
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# --- Training Setup ---

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 3 # Train for a few epochs to get a decent model, more for better accuracy

# MNIST Dataset (download if not available)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Initialize model, loss, and optimizer
model_to_train = DigitClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_to_train.parameters(), lr=LEARNING_RATE)

# --- Training Loop ---
print("\nStarting model training...")
for epoch in range(EPOCHS):
    model_to_train.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model_to_train(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch+1}/{EPOCHS}, Batch: {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')

# --- Evaluation (optional, but good practice) ---
model_to_train.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        outputs = model_to_train(data)
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f'\nTraining complete! Accuracy on test set: {(100 * correct / total):.2f}%')

# --- Save the trained model weights ---
model_filename = 'mnist_cnn.pth'
torch.save(model_to_train.state_dict(), model_filename)
print(f'Model weights saved to {model_filename}')


Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 49.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 2.16MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 16.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.51MB/s]



Starting model training...
Epoch: 1/3, Batch: 0/938, Loss: 2.3011
Epoch: 1/3, Batch: 100/938, Loss: 0.1424
Epoch: 1/3, Batch: 200/938, Loss: 0.1378
Epoch: 1/3, Batch: 300/938, Loss: 0.0386
Epoch: 1/3, Batch: 400/938, Loss: 0.0258
Epoch: 1/3, Batch: 500/938, Loss: 0.0570
Epoch: 1/3, Batch: 600/938, Loss: 0.1177
Epoch: 1/3, Batch: 700/938, Loss: 0.2933
Epoch: 1/3, Batch: 800/938, Loss: 0.1912
Epoch: 1/3, Batch: 900/938, Loss: 0.0454
Epoch: 2/3, Batch: 0/938, Loss: 0.0073
Epoch: 2/3, Batch: 100/938, Loss: 0.0511
Epoch: 2/3, Batch: 200/938, Loss: 0.0227
Epoch: 2/3, Batch: 300/938, Loss: 0.0149
Epoch: 2/3, Batch: 400/938, Loss: 0.0057
Epoch: 2/3, Batch: 500/938, Loss: 0.0652
Epoch: 2/3, Batch: 600/938, Loss: 0.0012
Epoch: 2/3, Batch: 700/938, Loss: 0.1557
Epoch: 2/3, Batch: 800/938, Loss: 0.1205
Epoch: 2/3, Batch: 900/938, Loss: 0.0050
Epoch: 3/3, Batch: 0/938, Loss: 0.0004
Epoch: 3/3, Batch: 100/938, Loss: 0.0176
Epoch: 3/3, Batch: 200/938, Loss: 0.0009
Epoch: 3/3, Batch: 300/938, Loss: 0